# Predictive Maintenance Classification Pipeline

This notebook is a guided implementation template for the Module 1 assessment.  
It follows the required execution order: EDA, data preparation, feature engineering, train/test split, training-set-only resampling, KNN scaling, Decision Tree without scaling, hyperparameter comparison, overfitting analysis, and final accuracy-based verdict.

Renan de Brito Leme

## 0. Environment Setup

In [ ]:
# ============================================================
# RF00 - Libraries import
# ============================================================

import numpy as np
import pandas as pd
import os

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Resampling
from imblearn.over_sampling import SMOTE
# Alternative if SMOTE is too slow or unavailable:
# from imblearn.under_sampling import RandomUnderSampler

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
sns.set_context("notebook")

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Environment configured successfully.")

## 1. Load the Dataset

In [ ]:
DATA_PATH = "predictive_maintenance.csv"
raw_dataset = pd.read_csv(DATA_PATH)
print("Dataset loaded successfully.")
print("Shape:", raw_dataset.shape)
raw_dataset.head()

## 2. Data Dictionary and Modeling Assumptions

Important modeling assumptions:

1. `machine_failure` is the binary target variable.
2. The columns `tool_wear_failure`, `heat_dissipation_failure`, `power_failure`, `overstrain_failure`, and `random_failure` are specific failure causes and must not be used as predictors because they would leak information about the target.
3. `udi` and `product_id` are identifiers, not physical process variables, so they should also be excluded from the model.
4. The relevant predictors are equipment type and sensor/process variables.

In [ ]:
target_column = "machine_failure"

identifier_columns = ["unique_id", "product_id"]

leakage_columns = [
    "tool_wear_failure",
    "heat_dissipation_failure",
    "power_failure",
    "overstrain_failure",
    "random_failure",
]

numeric_sensor_features = [
    "air_temperature_k",
    "process_temperature_k",
    "rotational_speed_rpm",
    "torque_nm",
    "tool_wear_min",
]

categorical_features = ["machine_type"]

print("Target column:", target_column)
print("Identifier columns:", identifier_columns)
print("Leakage columns:", leakage_columns)
print("Numeric sensor features:", numeric_sensor_features)
print("Categorical features:", categorical_features)

# Phase 1 — Exploratory Data Analysis (EDA)

Required outputs:

- Dataset dimensions.
- Variable data types.
- Descriptive statistics with `.describe()`.
- At least three analytical plots.
- A written interpretation connecting the EDA findings to modeling decisions.

In [ ]:
print("Dataset shape:", raw_dataset.shape)
raw_dataset.info()
print("-------------------------------------------------------")
print("Data types:")
print(raw_dataset.dtypes)

In [ ]:
raw_dataset.describe().T

In [ ]:
# Missing values overview
missing_summary = pd.DataFrame({
    "missing_count": raw_dataset.isna().sum(),
    "missing_percent": raw_dataset.isna().mean() * 100
}).sort_values("missing_count", ascending=False)

missing_summary

In [ ]:
# Target distribution
target_distribution = raw_dataset[target_column].value_counts().sort_index()

target_percentage = (
    raw_dataset[target_column]
    .value_counts(normalize=True)
    .sort_index() * 100
)

target_summary = pd.DataFrame({
    "count": target_distribution,
    "percentage": target_percentage.round(2),
})

target_summary

In [ ]:
# Plot 1: target imbalance
plt.figure(figsize=(6, 4))
sns.countplot(data=raw_dataset, x=target_column)
plt.title("Target Variable Distribution")
plt.xlabel("Machine Failure: 0 = Normal, 1 = Failure")
plt.ylabel("Count")
output_dir="outputs"
file_path = os.path.join(output_dir, "target_variable_distribution.png")
plt.savefig(file_path, dpi=150)
plt.show()
plt.close()
print(f"  Chart exported: {file_path}")

In [ ]:
# Plot 2: numerical variable distributions
raw_dataset[numeric_sensor_features].hist(figsize=(12, 8), bins=30)
plt.suptitle("Distribution of Numerical Predictive Variables")
plt.tight_layout()
output_dir="outputs"
file_path = os.path.join(output_dir, "distribution_of_numerical_predictive_variables.png")
plt.savefig(file_path, dpi=150)
plt.show()
plt.close()
print(f"  Chart exported: {file_path}")

In [ ]:
# Plot 3: Pearson correlation heatmap
correlation_columns = numeric_sensor_features + [target_column]
correlation_matrix = raw_dataset[correlation_columns].corr(numeric_only=True)

plt.figure(figsize=(9, 6))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Pearson Correlation Heatmap")
plt.tight_layout()
output_dir="outputs"
file_path = os.path.join(output_dir, "pearson_correlation_heatmap.png")
plt.savefig(file_path, dpi=150)
plt.show()
plt.close()
print(f"  Chart exported: {file_path}")

In [ ]:
# Plot 4: equipment type vs target
plt.figure(figsize=(7, 4))
sns.countplot(
    data=raw_dataset,
    x="machine_type",
    hue=target_column
)
plt.title("Machine Type vs Machine Failure")
plt.xlabel("Machine Type")
plt.ylabel("Count")
plt.legend(title="Failure")
plt.tight_layout()
output_dir="outputs"
file_path = os.path.join(output_dir, "machine_type_vs_machine_failure.png")
plt.savefig(file_path, dpi=150)
plt.show()
plt.close()
print(f"  Chart exported: {file_path}")